# X6Y3 Ramsey and ping-pong tune-up

Run on the existing Huracan service using the supplied X6Y3 configuration. This follows `TuneUpExample.ipynb`: Ramsey offsets 10, 1 and 0.1 MHz with 0.3, 3 and 30 µs windows, then amplitude error amplification. Native X90 and X remain independent; the generic drive collection's X/2 assumption does not apply here.

Hardware execution is opt-in. Preserve the input YAML, notebook and ZIP. No board installation, firmware loading, service changes or persistent RF changes are needed. Inspect each fit; a three-point amplitude interpolation is a candidate until an independent scan validates it. IQ projection is not a calibrated population or gate-fidelity measurement.


In [ ]:
import os, sys, json
from pathlib import Path
from datetime import datetime, timezone
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
LEEQ_ROOT = Path('/local/data/projects/LeeQ-lbnl-qubic')
HARNESS_ROOT = Path('/local/data/projects/qubic_validation')
sys.path.insert(0, str(LEEQ_ROOT))
sys.path.append(str(HARNESS_ROOT / 'artifacts/x6y3/venv/lib/python3.11/site-packages'))
RUN_HARDWARE = False
QUBIT = 0
if not RUN_HARDWARE:
    import socket
    def blocked(*args, **kwargs):
        raise RuntimeError('Offline notebook: network disabled')
    socket.socket.connect = socket.socket.connect_ex = socket.create_connection = blocked


In [ ]:
from leeq.setups.x6y3 import X6Y3Calibration
from leeq.setups.huracan import create_huracan_setup
from leeq.experiments.experiments import ExperimentManager
from leeq.experiments.x6y3 import acquire_plan, readout_plan, analyze_readout
from leeq.experiments.x6y3_tuneup import (
    RAMSEY_STAGES, leeq_ramsey_plan, pingpong_plan, fit_ramsey, fit_pingpong, save_candidate,
)
from leeq.experiments.x6y3_validation import verify_plan_parity
calibration = X6Y3Calibration(HARNESS_ROOT / 'X6Y3/configs_new/config.yaml')
huracan = create_huracan_setup('/local/data/artifacts/zcu216/20260903T_aa01c78f_rdrv230_d4_candidate/bits_aa01c78f/channel_config.json')
center_offset_mhz = 0.0
x90_scale = 1.0
output = HARNESS_ROOT / 'artifacts/x6y3/tuneup-runs' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')


In [ ]:
# Representative independent qcal schedule/waveform parity; no RPC.
for stage in range(3):
    plan = leeq_ramsey_plan(QUBIT, stage)
    plan.points = [plan.points[0], plan.points[17], plan.points[-1]]
    assert verify_plan_parity(calibration, huracan, plan)['pass']
for gate, counts in [('X90', [0, 4, 8, 12]), ('X', [0, 2, 4, 6])]:
    plan = pingpong_plan(QUBIT, gate, [0.99, 1.01], counts, blocks=1)
    assert verify_plan_parity(calibration, huracan, plan)['pass']
print('Offline protocol parity passed')


In [ ]:
if RUN_HARDWARE:
    import socket
    socket.setdefaulttimeout(60)
    ExperimentManager().register_setup(huracan)
    result = acquire_plan(calibration, huracan, readout_plan(QUBIT, shots=1024), output / 'readout')
    readout_report = analyze_readout(result)
    (output / 'readout' / 'analysis.json').write_text(json.dumps(readout_report, indent=2))


## Three-stage Ramsey

Use physical drive frequency `baseline + center + set_offset`, with X90–idle–negative X90. Update the center by `set_offset - fitted_frequency`. Fits must pass physical/quality guards before the next scan. The fit uncertainty does not include drift. Recheck the fine scan after amplitude tuning.


In [ ]:
ramsey_reports = []
if RUN_HARDWARE:
    for stage in range(3):
        plan = leeq_ramsey_plan(QUBIT, stage, center_offset_mhz=center_offset_mhz,
                                x90_scale=x90_scale, shots=1024)
        result = acquire_plan(calibration, huracan, plan, output / f'ramsey-stage{stage}')
        report = fit_ramsey(result, readout_report)
        (result['output_dir'] / 'analysis.json').write_text(json.dumps(report, indent=2))
        ramsey_reports.append(report)
        print(stage, report['new_center_offset_mhz'], report['frequency_std_mhz'], report['accepted'])
        if not report['accepted']:
            raise RuntimeError('Inspect the saved Ramsey data before proceeding')
        center_offset_mhz = report['new_center_offset_mhz']


## Independent X90 / X ping-pong

Repeat X an even number of times, or X90 a multiple of four, followed by a fixed X90. Add an opposite-phase final X90 control and randomize repeated blocks to reduce readout-offset and drift bias. Fit the phase-paired signal versus repetition count, then find the amplitude where its slope crosses zero. Start near the notebook amplitude; narrow the bracket and increase repetitions after inspecting the curves. Keep angles small enough for a local linear fit.

Run X90 first. Set `x90_scale` to its independently validated value before tuning X. Rerunning this cell requires a new `scan_name`. Validate the candidate and original amplitude in a fresh randomized scan, using one or two scales; these return slopes without another interpolation. A candidate needs a small residual slope and reproducible improvement, not merely an in-range interpolation.


In [ ]:
gate = 'X90'
scales = [0.98, 1.0, 1.02]
counts = [0, 4, 8, 12] if gate == 'X90' else [0, 2, 4, 6]
scan_name = 'X90-initial'
if RUN_HARDWARE:
    plan = pingpong_plan(QUBIT, gate, scales, counts,
                         center_offset_mhz=center_offset_mhz, x90_scale=x90_scale,
                         shots=1024, blocks=2)
    result = acquire_plan(calibration, huracan, plan, output / scan_name)
    report = fit_pingpong(result, readout_report)
    (result['output_dir'] / 'analysis.json').write_text(json.dumps(report, indent=2))
    print({key: value for key, value in report.items() if key != 'mean'})


## Save reviewed settings locally

After independent validation, fill in the settings and set `SAVE_VALIDATED_CANDIDATE=True`. This writes a new host-side YAML; it never overwrites the supplied configuration. Keep the corresponding raw IQ, manifests, fit reports, validation evidence and board checks alongside it.


In [ ]:
SAVE_VALIDATED_CANDIDATE = False
validated_settings = {}  # e.g. {0: {'frequency_offset_mhz': ..., 'X90_scale': ..., 'X_scale': ...}}
if SAVE_VALIDATED_CANDIDATE:
    if not validated_settings:
        raise ValueError('Enter independently validated settings first')
    save_candidate(calibration, validated_settings, output / 'candidate-config.yaml')
